##Configure all Bronze and Silver table names

In [0]:

CATALOG = "capretailnew"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

# Bronze source tables
BRONZE_SALES = f"{BRONZE}.bronze_sales_order_lines"
BRONZE_INVENTORY = f"{BRONZE}.bronze_inventory_movements"
BRONZE_RETURNS = f"{BRONZE}.bronze_sales_returns"

BRONZE_PRODUCT = f"{BRONZE}.bronze_product_master"
BRONZE_STORE = f"{BRONZE}.bronze_store_master"
BRONZE_SUPPLIER = f"{BRONZE}.bronze_supplier_master"

# Silver output tables
SILVER_SALES = f"{SILVER}.silver_sales_order_lines"
SILVER_INVENTORY = f"{SILVER}.silver_inventory_movements"
SILVER_RETURNS = f"{SILVER}.silver_sales_returns"

SILVER_PRODUCT = f"{SILVER}.silver_product_master"
SILVER_STORE = f"{SILVER}.silver_store_master"
SILVER_SUPPLIER = f"{SILVER}.silver_supplier_master"

print("Bronze:", BRONZE)
print("Silver:", SILVER)
print("Configuration completed")

##Deduplicate sales records

In [0]:

from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Bronze sales source (duplicate pattern was profiled in the Bronze layer:
# batch_010.csv is a replay of original batches)
sales_df = spark.table(BRONZE_SALES)

# Business columns (ingestion metadata excluded)
business_columns = [
    "order_line_id",
    "order_id",
    "order_ts",
    "store_id",
    "product_id",
    "customer_id",
    "quantity",
    "unit_price",
    "discount_pct",
    "sales_channel",
    "order_status",
    "tax_amount"
]

# Define deterministic ranking:
# Original batches are preferred over batch_010
dedup_window = (
    Window
    .partitionBy(*business_columns)
    .orderBy(
        when(
            col("source_file").contains("batch_010.csv"), 1
        ).otherwise(0).asc(),
        col("source_file").asc()
    )
)

# Assign a rank to each business-level record
sales_ranked = sales_df.withColumn(
    "_duplicate_rank",
    row_number().over(dedup_window)
)

# Keep one original record per business record
sales_business_dedup = (
    sales_ranked
    .filter(col("_duplicate_rank") == 1)
    .drop("_duplicate_rank")
)

# Preserve replayed duplicate records separately
sales_dedup_audit = (
    sales_ranked
    .filter(col("_duplicate_rank") > 1)
    .drop("_duplicate_rank")
)

# Display reconciliation
print("Original Bronze records:", sales_df.count())
print("Clean sales records:", sales_business_dedup.count())
print("Duplicate records in audit:", sales_dedup_audit.count())

print(
    "Remaining duplicate business groups:",
    sales_business_dedup
    .groupBy(*business_columns)
    .count()
    .filter(col("count") > 1)
    .count()
)

display(sales_dedup_audit.select(
    "order_line_id",
    "source_file",
    "ingestion_timestamp"
))

##Cast sales columns and summarize data quality

In [0]:
from pyspark.sql.functions import (
    col, trim, when, count, min, max,
    expr, sum, countDistinct
)

# 1. Safely test conversions without changing the source DataFrame
sales_checked = (
    sales_business_dedup
    .withColumn(
        "quantity_check",
        expr("try_cast(trim(quantity) AS INT)")
    )
    .withColumn(
        "unit_price_check",
        expr("try_cast(trim(unit_price) AS DECIMAL(18,2))")
    )
    .withColumn(
        "discount_pct_check",
        expr("try_cast(trim(discount_pct) AS DECIMAL(5,2))")
    )
    .withColumn(
        "tax_amount_check",
        expr("try_cast(trim(tax_amount) AS DECIMAL(18,2))")
    )
    .withColumn(
        "order_ts_check",
        expr("try_cast(order_ts AS TIMESTAMP)")
    )
)

# 2. Check missing and invalid values
sales_quality_summary = sales_checked.agg(
    count("*").alias("total_records"),

    sum(when(
        col("order_line_id").isNull() |
        (trim(col("order_line_id")) == ""), 1
    ).otherwise(0)).alias("missing_order_line_id"),

    sum(when(
        col("order_id").isNull() |
        (trim(col("order_id")) == ""), 1
    ).otherwise(0)).alias("missing_order_id"),

    sum(when(
        col("customer_id").isNull() |
        (trim(col("customer_id")) == ""), 1
    ).otherwise(0)).alias("missing_customer_id"),

    sum(when(col("quantity_check").isNull(), 1)
        .otherwise(0)).alias("invalid_quantity"),

    sum(when(col("unit_price_check").isNull(), 1)
        .otherwise(0)).alias("invalid_unit_price"),

    sum(when(col("discount_pct_check").isNull(), 1)
        .otherwise(0)).alias("invalid_discount_pct"),

    sum(when(col("tax_amount_check").isNull(), 1)
        .otherwise(0)).alias("invalid_tax_amount"),

    sum(when(col("order_ts_check").isNull(), 1)
        .otherwise(0)).alias("invalid_order_timestamp")
)

display(sales_quality_summary)

##Build row-level DQ flags

In [0]:
from pyspark.sql.functions import (
    col, trim, when, lit, concat_ws, array,
    size, filter as array_filter
)

sales_dq_tagged = (
    sales_checked

    # Quantity validation
    .withColumn(
        "dq_invalid_quantity",
        col("quantity_check").isNull() |
        (col("quantity_check") <= 0)
    )

    # Unit price validation
    .withColumn(
        "dq_invalid_unit_price",
        col("unit_price_check").isNull() |
        (col("unit_price_check") == 999999.99) |
        (col("unit_price_check") <= 0)
    )

    # Discount validation
    .withColumn(
        "dq_invalid_discount",
        col("discount_pct_check").isNull() |
        (col("discount_pct_check") < 0) |
        (col("discount_pct_check") > 100)
    )

    # Required relationship candidate
    .withColumn(
        "dq_missing_order_id",
        col("order_id").isNull() |
        (trim(col("order_id")) == "")
    )

    # Missing attributes to review
    .withColumn(
        "dq_missing_customer",
        col("customer_id").isNull() |
        (trim(col("customer_id")) == "")
    )
    .withColumn(
        "dq_missing_sales_channel",
        col("sales_channel").isNull() |
        (trim(col("sales_channel")) == "")
    )
    .withColumn(
        "dq_missing_order_status",
        col("order_status").isNull() |
        (trim(col("order_status")) == "")
    )
)

# Build a readable list of DQ reasons for each row
sales_dq_tagged = sales_dq_tagged.withColumn(
    "dq_reasons",
    concat_ws(
        ", ",
        when(col("dq_invalid_quantity"), lit("INVALID_QUANTITY")),
        when(col("dq_invalid_unit_price"), lit("INVALID_UNIT_PRICE")),
        when(col("dq_invalid_discount"), lit("INVALID_DISCOUNT")),
        when(col("dq_missing_order_id"), lit("MISSING_ORDER_ID")),
        when(col("dq_missing_customer"), lit("MISSING_CUSTOMER")),
        when(col("dq_missing_sales_channel"), lit("MISSING_SALES_CHANNEL")),
        when(col("dq_missing_order_status"), lit("MISSING_ORDER_STATUS"))
    )
)

# Records with no DQ flags
sales_dq_tagged = sales_dq_tagged.withColumn(
    "has_dq_issue",
    col("dq_reasons") != ""
)

print("Total records:", sales_dq_tagged.count())
print(
    "Records with at least one DQ issue:",
    sales_dq_tagged.filter(col("has_dq_issue")).count()
)
print(
    "Records without these DQ issues:",
    sales_dq_tagged.filter(~col("has_dq_issue")).count()
)

In [0]:
dq_columns = [
    "dq_invalid_quantity",
    "dq_invalid_unit_price",
    "dq_invalid_discount",
    "dq_missing_order_id",
    "dq_missing_customer",
    "dq_missing_sales_channel",
    "dq_missing_order_status"
]

for dq_column in dq_columns:
    print(
        dq_column,
        sales_dq_tagged.filter(col(dq_column)).count()
    )

In [0]:
display(
    sales_dq_tagged
    .filter(col("has_dq_issue"))
    .select(
        "order_line_id",
        "order_id",
        "customer_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "sales_channel",
        "order_status",
        "dq_reasons",
        "source_file"
    )
    .orderBy("order_line_id")
)

##Separate valid and invalid sales records

In [0]:
from pyspark.sql.functions import col, trim

# 1. Identify hard-invalid records
sales_hard_invalid = (
    col("dq_invalid_quantity") |
    col("dq_invalid_unit_price") |
    col("dq_invalid_discount") |
    col("dq_missing_order_id")
)

# 2. Identify records with missing order_status
sales_missing_status = (
    col("dq_missing_order_status")
)

# 3. Combine all quarantine conditions
sales_quarantine_condition = (
    sales_hard_invalid | sales_missing_status
)

# 4. Create clean Sales Silver DataFrame
sales_silver = (
    sales_dq_tagged
    .filter(~sales_quarantine_condition)
    .select(
        col("order_line_id"),
        col("order_id"),
        col("order_ts_check").alias("order_ts"),
        col("store_id"),
        col("product_id"),
        col("customer_id"),
        col("quantity_check").alias("quantity"),
        col("unit_price_check").alias("unit_price"),
        col("discount_pct_check").alias("discount_pct"),
        col("sales_channel"),
        col("order_status"),
        col("tax_amount_check").alias("tax_amount"),
        col("source_file"),
        col("ingestion_timestamp")
    )
)

# 5. Move all invalid records into quarantine
sales_quarantine = (
    sales_dq_tagged
    .filter(sales_quarantine_condition)
    .withColumn(
        "quarantine_reason",
        col("dq_reasons")
    )
)

# 6. Preserve duplicate records in the audit table
sales_duplicate_audit = sales_dedup_audit

# 7. Write clean Sales Silver table
(sales_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("capretailnew.silver.silver_sales_order_lines")
)

# 8. Write Sales quarantine table
(sales_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("capretailnew.silver.quarantine_sales_order_lines")
)

# 9. Write duplicate audit table
(sales_duplicate_audit.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("capretailnew.silver.sales_order_lines_duplicate_audit")
)

# 10. Final validation
print("SALES SILVER:", sales_silver.count())
print("SALES QUARANTINE:", sales_quarantine.count())
print("DUPLICATE AUDIT:", sales_duplicate_audit.count())

print(
    "NULL ORDER STATUS IN SILVER:",
    sales_silver.filter(
        col("order_status").isNull() |
        (trim(col("order_status")) == "")
    ).count()
)

print(
    "MISSING STATUS RECORDS IN QUARANTINE:",
    sales_quarantine.filter(
        col("dq_missing_order_status") == True
    ).count()
)

print(
    "TOTAL RECONCILIATION:",
    sales_silver.count()
    + sales_quarantine.count()
    + sales_duplicate_audit.count()
)

print("Sales Silver tables overwritten successfully.")

## Validate the saved tables

In [0]:
from pyspark.sql.functions import col
silver_sales = spark.table(
    "capretailnew.silver.silver_sales_order_lines"
)

quarantine_sales = spark.table(
    "capretailnew.silver.quarantine_sales_order_lines"
)

duplicate_audit = spark.table(
    "capretailnew.silver.sales_order_lines_duplicate_audit"
)

print("Silver count:", silver_sales.count())
print("Quarantine count:", quarantine_sales.count())
print("Duplicate audit count:", duplicate_audit.count())

print(
    "Duplicate business keys remaining:",
    silver_sales.groupBy("order_line_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Silver schema:")
silver_sales.printSchema()